# 第 5 章：训练模型、生成文本、保存权重

默认使用小模型和短文本，便于在 Mac 上跑完整流程。后面提供 PDF 中较大规模的
GPT-2 small 训练设置，以及加载已有 GPT-2 权重的可选单元；它们默认关闭。


In [ ]:
from pathlib import Path
import sys

# notebook 在项目根目录或章节目录中打开时，都能找到我们的 Python 文件。
ROOT = Path.cwd()
if not (ROOT / "chapter02").exists():
    ROOT = ROOT.parent
if not (ROOT / "chapter02").exists():
    raise FileNotFoundError("请从本项目的根目录或章节目录启动 Jupyter")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目目录:", ROOT)


## 1. 准备输入与“右移一格”的答案

每个输入位置预测它后面的 token。先用短文本反复出现，让小模型能在几步之内看到例子。
要用原书的 `the-verdict.txt`，看下面 `RUN_BOOK_EXPERIMENT` 的单元。


In [ ]:
import torch
import tiktoken
from chapter02.data import SAMPLE_TEXT, load_verdict, create_dataloader_v1
from chapter04.gpt import GPTModel, TINY_CONFIG, GPT_CONFIG_124M
from chapter05.training import (
    choose_device, calc_loss_batch, calc_loss_loader, train_model_simple,
    text_to_token_ids, token_ids_to_text, generate, plot_losses,
    save_checkpoint, load_checkpoint
)

torch.manual_seed(123)
tokenizer = tiktoken.get_encoding("gpt2")
text = SAMPLE_TEXT * 80
split = int(0.9 * len(text))
train_text, val_text = text[:split], text[split:]
cfg = TINY_CONFIG.copy()
cfg["context_length"] = 16
train_loader = create_dataloader_v1(
    train_text, batch_size=4, max_length=16, stride=16,
    drop_last=True, tokenizer=tokenizer
)
val_loader = create_dataloader_v1(
    val_text, batch_size=4, max_length=16, stride=16,
    shuffle=False, drop_last=False, tokenizer=tokenizer
)
x, y = next(iter(train_loader))
print("输入:", x.shape, x[0, :6].tolist())
print("目标:", y.shape, y[0, :6].tolist())


## 2. 从 `[B,T,V]` 算一个 loss

`V` 是词表大小。把 `B` 和 `T` 合并为 `B×T` 道选择题，
用交叉熵比较每题的 logits 和正确 token id。loss 越小，表示预测越准确。
注意力里的 softmax 决定“参考哪些旧 token”；这里的交叉熵评估“下一个 token 猜得好不好”。


In [ ]:
device = choose_device()
model = GPTModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
initial_loss = calc_loss_batch(x, y, model, device)
print("设备:", device, "训练前 loss:", round(initial_loss.item(), 3))
print("输入位置数:", x.numel(), "每位置都产生一次预测")


## 3. 小模型训练几步

每一步依次 `zero_grad → forward/loss → backward → optimizer.step`。
`backward` 把梯度经过输出层、block、注意力一直传回 embedding。
训练和验证分开算 loss，避免只会背训练文本却不知道效果如何。


In [ ]:
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=1, eval_freq=2, eval_iter=1, max_steps=4
)
print("训练 loss:", [round(n, 3) for n in train_losses])
print("验证 loss:", [round(n, 3) for n in val_losses])
print("已看过的 token 数:", tokens_seen)


训练 loss 与验证 loss 最好一起看。若训练 loss 降了、验证 loss 却升了，
模型可能只是在背训练文本。这里仅训练 4 步，图上的点很少，主要用来学习画图方法。


In [ ]:
loss_figure = plot_losses(tokens_seen, train_losses, val_losses)
loss_figure


## 4. 用温度和 top-k 生成

`temperature=0` 总选最大 logits；正温度会抽样。`top_k=20` 先只保留得分最高的 20 个候选。
模型只训练了几步，生成文本没有意义是正常的；这个单元要观察的是自回归的代码流程。


In [ ]:
model.eval()
start = text_to_token_ids("The fox", tokenizer).to(device)
sampled = generate(
    model, start, max_new_tokens=10,
    context_size=cfg["context_length"], temperature=0.8, top_k=20
)
print(token_ids_to_text(sampled.cpu(), tokenizer))


## 5. 保存和恢复

保存的是训练后参数；如果想接着训练，还要保存 AdamW 的状态。
恢复后在 `eval()` 模式比较同一个输入，两个模型应得到相同 logits。


In [ ]:
checkpoint_path = ROOT / "chapter05" / "checkpoints" / "tiny_gpt.pt"
save_checkpoint(checkpoint_path, model, optimizer, note="第 5 章小模型练习")
restored = GPTModel(cfg).to(device)
restored_optimizer = torch.optim.AdamW(
    restored.parameters(), lr=4e-4, weight_decay=0.1
)
metadata = load_checkpoint(checkpoint_path, restored, restored_optimizer, device)
restored.eval()
print("保存位置:", checkpoint_path)
print("备注:", metadata)
print("恢复后预测一致:", torch.allclose(model(start), restored(start)))


## 6. 可选：按 PDF 的规模训练

这会读取 `the-verdict.txt`，建立 `context_length=256`、`batch_size=2` 的数据集，
并训练 GPT-2 small 结构 10 轮。书中叫它 124M，但由于这里输入与输出词表表项没有共享，
实际可训练参数约 163M。Mac 18GB 可尝试运行，但速度会慢，且短小说易过拟合。


In [ ]:
RUN_BOOK_EXPERIMENT = False
if RUN_BOOK_EXPERIMENT:
    book_text = load_verdict()
    split = int(0.9 * len(book_text))
    book_cfg = GPT_CONFIG_124M.copy()
    book_cfg["context_length"] = 256
    book_train = create_dataloader_v1(
        book_text[:split], batch_size=2, max_length=256, stride=256,
        drop_last=True, tokenizer=tokenizer
    )
    book_val = create_dataloader_v1(
        book_text[split:], batch_size=2, max_length=256, stride=256,
        shuffle=False, drop_last=False, tokenizer=tokenizer
    )
    book_model = GPTModel(book_cfg).to(device)
    book_optimizer = torch.optim.AdamW(
        book_model.parameters(), lr=4e-4, weight_decay=0.1
    )
    book_train_loss, book_val_loss, book_tokens = train_model_simple(
        book_model, book_train, book_val, book_optimizer, device,
        num_epochs=10, eval_freq=5, eval_iter=5
    )
    print("最后的训练和验证 loss:", book_train_loss[-1], book_val_loss[-1])
else:
    print("完整规模训练已跳过；小模型流程已在上面运行。")


## 7. 可选：加载已经预训练的 GPT-2

书里从 OpenAI TensorFlow checkpoint 读取权重；本项目改用 Hugging Face Transformers。
必须使模型配置与 GPT-2 small 匹配：12 层、12 头、宽度 768、位置表 1024，并设 `qkv_bias=True`。
首次下载需要数百 MB。普通推理中权重固定；若选择微调，权重还会更新。


In [ ]:
RUN_PRETRAINED = False
if RUN_PRETRAINED:
    from chapter05.pretrained import load_pretrained_gpt2

    pretrained_cfg = GPT_CONFIG_124M.copy()
    pretrained_cfg.update(qkv_bias=True, drop_rate=0.0)
    pretrained = GPTModel(pretrained_cfg)  # 先在 CPU 上复制权重
    load_pretrained_gpt2(pretrained, model_name="gpt2")
    pretrained = pretrained.to(device).eval()
    first_head_q = pretrained.trf_blocks[0].att.W_query.weight[
        :pretrained.trf_blocks[0].att.head_dim
    ]
    print("第 1 层第 1 头的 Q 权重形状:", first_head_q.shape)
    start = text_to_token_ids("Every effort moves you", tokenizer).to(device)
    output = generate(pretrained, start, 20, 1024, temperature=0.8, top_k=40)
    print(token_ids_to_text(output.cpu(), tokenizer))
else:
    print("预训练权重下载已跳过。")
